# Alchemy GeomML + TDA — Финальные графики

Три эксперимента:
1. **bs=256** — baselines: FCNN, SchNet, EGNN
2. **bs=512** — только EGNN (для сравнения batch size; остальные модели v29 имели баги)
3. **bs=1024** — все EGNN-варианты: EGNN, EGNN+TDA, EGNN Vector, EGNN Vector+TDA, EGNN Tensor, EGNN Tensor+TDA

In [ ]:
import os
if not os.path.exists('alchemy-geom-tda'):
    !git clone https://github.com/ThomasMoore25/alchemy-geom-tda.git
%cd alchemy-geom-tda
!git pull

In [ ]:
import glob, numpy as np, pandas as pd
import matplotlib.pyplot as plt

FIG = 'results/figures/final'
os.makedirs(FIG, exist_ok=True)

# Загрузка всех history
def load_history(path):
    df = pd.read_csv(path)
    return df

# Эксперимент 1: bs=256
e1 = {}
for f in glob.glob('results/experiments/batch_size_256/history_*.csv'):
    name = os.path.basename(f).replace('history_', '').replace('_all.csv', '').replace('.csv', '')
    e1[name] = load_history(f)
    print(f'E1 bs=256 {name}: {len(e1[name])} epochs')

# Эксперимент 2: bs=512 (только EGNN)
e2 = {}
for f in glob.glob('results/experiments/batch_size_512/history_egnn_all_*.csv'):
    e2['egnn'] = load_history(f)
    print(f'E2 bs=512 egnn: {len(e2["egnn"])} epochs')

# Эксперимент 3: bs=1024 — финальные метрики
e3_results = {
    'EGNN':           {'mu': 0.284, 'alpha': 0.583, 'gap': 0.0058, 'loss': 0.344, 'epochs': 135, 'params': 951759, 'time_s': 12500},
    'EGNN+TDA':       {'mu': 0.298, 'alpha': 0.619, 'gap': 0.0061, 'loss': 0.362, 'epochs': 146, 'params': 971831, 'time_s': 13900},
    'EGNN Vector':    {'mu': 4.123, 'alpha': 0.354, 'gap': 0.0044, 'loss': 0.167, 'epochs': 188, 'params': 968023, 'time_s': 18975},
    'EGNN Vec+TDA':   {'mu': 4.121, 'alpha': 0.510, 'gap': 0.0055, 'loss': 0.216, 'epochs': 108, 'params': 981439, 'time_s': 11279},
    'EGNN Tensor':    {'mu': 4.121, 'alpha': 0.957, 'gap': 0.0091, 'loss': 0.394, 'epochs': 475, 'params': 941900, 'time_s': 43100},
    'EGNN Ten+TDA':   {'mu': 4.111, 'alpha': 1.011, 'gap': 0.0104, 'loss': 0.428, 'epochs': 273, 'params': 948660, 'time_s': 29317},
}
print(f'E3 bs=1024: {len(e3_results)} models (from Kaggle logs)')

# v26.1 test results
e1_test = {}
for name, df in e1.items():
    last = df.iloc[-1]
    e1_test[name.upper()] = {'mu': last['test_mu_mae'], 'alpha': last['test_alpha_mae'], 
                             'gap': last['test_gap_mae'], 'loss': last['test_loss'], 'epochs': len(df)}
    print(f'  {name.upper()}: mu={last["test_mu_mae"]:.3f}, alpha={last["test_alpha_mae"]:.3f}, gap={last["test_gap_mae"]:.4f}')

## Эксперимент 1: Baselines (bs=256) — Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12), constrained_layout=True)
fig.suptitle('Эксперимент 1: Baselines (bs=256, полный датасет)', fontsize=16, fontweight='bold')

colors = {'FCNN': '#607D8B', 'SCHNET': '#795548', 'EGNN': '#2196F3'}

for ax, (col, title) in zip(axes.flat, [
    ('val_loss', 'Val Loss'),
    ('val_mu_mae', 'μ MAE (Debye)'),
    ('val_alpha_mae', 'α MAE (Bohr³)'),
    ('val_gap_mae', 'gap MAE (Hartree)')
]):
    for name, df in e1.items():
        label = name.upper()
        if col in df.columns:
            ax.plot(df['epoch'], df[col], color=colors[label], label=label, linewidth=2)
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('Epoch', fontsize=12)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

plt.savefig(f'{FIG}/01_baselines_training.png', dpi=150, facecolor='white')
plt.show()
print(f'Saved: {FIG}/01_baselines_training.png')

## Эксперимент 1: Baselines — Test Metrics

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5), constrained_layout=True)
fig.suptitle('Эксперимент 1: Test Metrics — FCNN vs SchNet vs EGNN (bs=256)', fontsize=14, fontweight='bold')

models = list(e1_test.keys())
colors = ['#607D8B', '#795548', '#2196F3']

for ax, (metric, title, unit) in zip(axes, [
    ('mu', 'μ MAE', 'Debye'),
    ('alpha', 'α MAE', 'Bohr³'),
    ('gap', 'gap MAE', 'Hartree'),
    ('loss', 'Test Loss', 'normalized')
]):
    vals = [e1_test[m][metric] for m in models]
    bars = ax.bar(models, vals, color=colors, edgecolor='black')
    ax.set_title(f'{title} ({unit})', fontsize=12)
    ax.grid(True, alpha=0.3, axis='y')
    best_idx = vals.index(min(vals))
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.savefig(f'{FIG}/02_baselines_test_metrics.png', dpi=150, facecolor='white')
plt.show()
print(f'Saved: {FIG}/02_baselines_test_metrics.png')

## Эксперимент 2: Batch Size Impact (EGNN only)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
fig.suptitle('Эксперимент 2: Влияние Batch Size на EGNN', fontsize=14, fontweight='bold')

# Left: Training curves
ax = axes[0]
ax.plot(e1['egnn']['epoch'], e1['egnn']['val_loss'], color='#2196F3', label='bs=256 (131 epochs)', linewidth=2)
ax.plot(e2['egnn']['epoch'], e2['egnn']['val_loss'], color='#FF9800', label='bs=512 (164 epochs)', linewidth=2)
ax.axhline(y=0.344, color='#4CAF50', linestyle='--', label='bs=1024 (135 epochs, test_loss=0.344)')
ax.set_title('Val Loss', fontsize=13)
ax.set_xlabel('Epoch', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Right: Bar chart
ax = axes[1]
bs_vals = ['bs=256', 'bs=512', 'bs=1024']
mu_vals = [e1_test['EGNN']['mu'], 0.245, 0.284]
alpha_vals = [e1_test['EGNN']['alpha'], 0.460, 0.583]
gap_vals = [e1_test['EGNN']['gap'], 0.0046, 0.0058]
x = np.arange(3)
w = 0.25
ax.bar(x - w, mu_vals, w, label='μ MAE (D)', color='#2196F3', edgecolor='black')
ax.bar(x, alpha_vals, w, label='α MAE (Bohr³)', color='#FF9800', edgecolor='black')
ax.bar(x + w, gap_vals, w, label='gap MAE (Ha)', color='#4CAF50', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(bs_vals)
ax.set_title('Test MAE by Batch Size', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.savefig(f'{FIG}/03_batch_size_impact.png', dpi=150, facecolor='white')
plt.show()
print(f'Saved: {FIG}/03_batch_size_impact.png')

## Эксперимент 3: Все EGNN-варианты (bs=1024) — Test Metrics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12), constrained_layout=True)
fig.suptitle('Эксперимент 3: Все EGNN-варианты (bs=1024, 2 GPU T4)', fontsize=16, fontweight='bold')

models = list(e3_results.keys())
colors = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0', '#F44336', '#E91E63']

for ax, (metric, title, unit) in zip(axes.flat, [
    ('mu', 'μ MAE (Debye)', 'D'),
    ('alpha', 'α MAE (Bohr³)', 'Bohr³'),
    ('gap', 'gap MAE (Hartree)', 'Ha'),
    ('loss', 'Test Loss', 'normalized')
]):
    vals = [e3_results[m][metric] for m in models]
    bars = ax.barh(models, vals, color=colors, edgecolor='black')
    ax.set_title(title, fontsize=13)
    ax.grid(True, alpha=0.3, axis='x')
    best_idx = vals.index(min(vals))
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', ha='left', va='center', fontsize=10, fontweight='bold')

plt.savefig(f'{FIG}/04_egnn_variants_test_metrics.png', dpi=150, facecolor='white')
plt.show()
print(f'Saved: {FIG}/04_egnn_variants_test_metrics.png')

## Эксперимент 3: Попарное сравнение EGNN vs EGNN+TDA

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5), constrained_layout=True)
fig.suptitle('EGNN vs EGNN+TDA (bs=1024) — TDA не помогает', fontsize=14, fontweight='bold')

metrics = [('mu', 'μ MAE (D)'), ('alpha', 'α MAE (Bohr³)'), ('gap', 'gap MAE (Ha)'), ('loss', 'Test Loss')]
egnn_vals = [e3_results['EGNN'][m] for m, _ in metrics]
tda_vals = [e3_results['EGNN+TDA'][m] for m, _ in metrics]

x = np.arange(4)
w = 0.35
for ax, (i, (m, title)) in zip(axes, enumerate(metrics)):
    ax.bar(['EGNN', 'EGNN+TDA'], [egnn_vals[i], tda_vals[i]], 
           color=['#2196F3', '#FF9800'], edgecolor='black')
    ax.set_title(title, fontsize=12)
    ax.grid(True, alpha=0.3, axis='y')
    delta = (tda_vals[i] - egnn_vals[i]) / egnn_vals[i] * 100
    ax.text(1, tda_vals[i] * 1.05, f'+{delta:.0f}%', ha='center', fontsize=12, color='#F44336', fontweight='bold')

plt.savefig(f'{FIG}/05_egnn_vs_tda.png', dpi=150, facecolor='white')
plt.show()
print(f'Saved: {FIG}/05_egnn_vs_tda.png')

## Эксперимент 3: Попарное сравнение EGNN vs EGNN Vector

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5), constrained_layout=True)
fig.suptitle('EGNN (скаляр μ) vs EGNN Vector (вектор μ) — Key Finding', fontsize=14, fontweight='bold')

egnn_vals = [e3_results['EGNN'][m] for m, _ in metrics]
vec_vals = [e3_results['EGNN Vector'][m] for m, _ in metrics]

for ax, (i, (m, title)) in zip(axes, enumerate(metrics)):
    ax.bar(['EGNN', 'EGNN Vector'], [egnn_vals[i], vec_vals[i]],
           color=['#2196F3', '#4CAF50'], edgecolor='black')
    ax.set_title(title, fontsize=12)
    ax.grid(True, alpha=0.3, axis='y')
    delta = (vec_vals[i] - egnn_vals[i]) / egnn_vals[i] * 100
    color = '#F44336' if delta > 0 else '#1a7f37'
    sign = '+' if delta > 0 else ''
    ax.text(1, vec_vals[i] * 1.05, f'{sign}{delta:.0f}%', ha='center', fontsize=12, color=color, fontweight='bold')

plt.savefig(f'{FIG}/06_egnn_vs_vector.png', dpi=150, facecolor='white')
plt.show()
print(f'Saved: {FIG}/06_egnn_vs_vector.png')

## Эксперимент 3: Попарное сравнение EGNN Vector vs EGNN Tensor

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5), constrained_layout=True)
fig.suptitle('EGNN Vector (α via MLP) vs EGNN Tensor (α via physics) — Часть B', fontsize=14, fontweight='bold')

vec_vals = [e3_results['EGNN Vector'][m] for m, _ in metrics]
ten_vals = [e3_results['EGNN Tensor'][m] for m, _ in metrics]

for ax, (i, (m, title)) in zip(axes, enumerate(metrics)):
    ax.bar(['EGNN Vector', 'EGNN Tensor'], [vec_vals[i], ten_vals[i]],
           color=['#4CAF50', '#F44336'], edgecolor='black')
    ax.set_title(title, fontsize=12)
    ax.grid(True, alpha=0.3, axis='y')
    delta = (ten_vals[i] - vec_vals[i]) / vec_vals[i] * 100
    color = '#F44336' if delta > 0 else '#1a7f37'
    sign = '+' if delta > 0 else ''
    ax.text(1, ten_vals[i] * 1.05, f'{sign}{delta:.0f}%', ha='center', fontsize=12, color=color, fontweight='bold')

plt.savefig(f'{FIG}/07_vector_vs_tensor.png', dpi=150, facecolor='white')
plt.show()
print(f'Saved: {FIG}/07_vector_vs_tensor.png')

## Эксперимент 3: Параметры и время обучения

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
fig.suptitle('Параметры и время обучения (bs=1024)', fontsize=14, fontweight='bold')

models_short = ['EGNN', 'EGNN+TDA', 'EGNN\nVector', 'EGNN\nVec+TDA', 'EGNN\nTensor', 'EGNN\nTen+TDA']
params = [e3_results[m]['params'] for m in e3_results]
times = [e3_results[m]['time_s'] / 3600 for m in e3_results]  # в часах
colors = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0', '#F44336', '#E91E63']

# Left: Parameters
ax = axes[0]
bars = ax.bar(models_short, params, color=colors, edgecolor='black')
ax.set_title('Параметры', fontsize=13)
ax.set_ylabel('Count', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, params):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
            f'{val:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Right: Training time
ax = axes[1]
bars = ax.bar(models_short, times, color=colors, edgecolor='black')
ax.set_title('Время обучения (часы)', fontsize=13)
ax.set_ylabel('Hours', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.1f}h', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.savefig(f'{FIG}/08_params_and_time.png', dpi=150, facecolor='white')
plt.show()
print(f'Saved: {FIG}/08_params_and_time.png')

## Эксперимент 3: TDA Analysis — почему 54% фичей всегда ноль

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)

cats = ['H₀ Betti\n(16 bins)', 'H₁ Betti\n(16 bins)', 'H₂ Betti\n(16 bins)', 'Entropy+Mean\n(4 dims)']
total = [16, 16, 16, 4]
always_zero = [0, 12, 16, 0]
nonzero = [t - z for t, z in zip(total, always_zero)]

x = range(len(cats))
ax.bar(x, nonzero, color='#4CAF50', label='Информативные', edgecolor='black')
ax.bar(x, always_zero, bottom=nonzero, color='#F44336', label='Всегда ноль', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(cats, fontsize=12)
ax.set_ylabel('Количество фичей', fontsize=13)
ax.set_title('TDA Feature Quality: 54% фичей всегда ноль\n(молекулы Alchemy 9-29 атомов слишком простые)', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.5, 0.95, '54% ALWAYS ZERO → BatchNorm → Шум → Ухудшение',
        transform=ax.transAxes, ha='center', fontsize=13, fontweight='bold', color='#F44336',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

plt.savefig(f'{FIG}/09_tda_feature_analysis.png', dpi=150, facecolor='white')
plt.show()
print(f'Saved: {FIG}/09_tda_feature_analysis.png')

## Сводная таблица всех результатов

In [ ]:
fig, ax = plt.subplots(figsize=(18, 8), constrained_layout=True)
ax.axis('off')

table_data = [
    ['Exp', 'Model', 'BS', 'Epochs', 'μ MAE', 'α MAE', 'gap MAE', 'Loss', 'Params'],
    ['1', 'FCNN', '256', '176', '0.851', '2.271', '0.026', '1.235', '53K'],
    ['1', 'SchNet', '256', '226', '0.131', '0.442', '0.003', '0.186', '320K'],
    ['1', 'EGNN', '256', '131', '0.179', '0.393', '0.004', '0.227', '952K'],
    ['2', 'EGNN', '512', '164', '0.245', '0.460', '0.005', '0.285', '952K'],
    ['3', 'EGNN', '1024', '135', '0.284', '0.583', '0.006', '0.344', '952K'],
    ['3', 'EGNN+TDA', '1024', '146', '0.298', '0.619', '0.006', '0.362', '972K'],
    ['3', 'EGNN Vector', '1024', '188', '4.123', '0.354', '0.004', '0.167', '968K'],
    ['3', 'EGNN Vec+TDA', '1024', '108', '4.121', '0.510', '0.006', '0.216', '981K'],
    ['3', 'EGNN Tensor*', '1024', '475', '4.121', '0.957', '0.009', '0.394', '942K'],
    ['3', 'EGNN Ten+TDA', '1024', '273', '4.111', '1.011', '0.010', '0.428', '949K'],
]

table = ax.table(cellText=table_data[1:], colLabels=table_data[0], cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.0, 1.8)

for j in range(9):
    table[0, j].set_facecolor('#37474F')
    table[0, j].set_text_props(color='white', fontweight='bold')

colors_row = ['#ECEFF1', '#F5F5DC', '#E3F2FD', '#FFF3E0', '#E8F5E9', '#FFF3E0', '#F3E5F5', '#FFF3E0', '#FFEBEE', '#FCE4EC']
for i, color in enumerate(colors_row):
    for j in range(9):
        table[i + 1, j].set_facecolor(color)

# Highlight best
for col, row in [(4, 1), (5, 2), (6, 1), (7, 7)]:  # SchNet mu, SchNet alpha, FCNN gap, EGNN Vector loss
    table[row, col].set_text_props(fontweight='bold', color='#2E7D32')

ax.set_title('Все результаты: 3 эксперимента, 8 моделей', fontsize=14, fontweight='bold', pad=20)
ax.text(0.5, -0.03, '* EGNN Tensor: 475 эпох (Kaggle 12h лимит). Полный прогон не завершён.',
        transform=ax.transAxes, ha='center', fontsize=10, style='italic', color='gray')

plt.savefig(f'{FIG}/10_summary_table.png', dpi=150, facecolor='white', bbox_inches='tight')
plt.show()
print(f'Saved: {FIG}/10_summary_table.png')